# Practice Report -- data notes and findings

Real investigation behind the dashboard's Practice Report tab
(`src/injury_report.py`), which wraps nflverse's `load_injuries()`
(`src/ingest.py::get_injuries`) into a per-week export.

Three questions this notebook answers with real data before anything got
built, per CLAUDE.md's non-negotiable against presenting derived values
with more confidence than the source supports:

1. How does nflverse's `practice_status`/`report_status` relate to the
   Sleeper `injury_status` already driving the Injury tab / Lineup Risks
   panel? Redundant, or genuinely different signals?
2. Is there a useful discrepancy between `report_primary_injury` and
   `practice_primary_injury`?
3. Is practice participation actually predictive -- does a "Limited"
   practice designation correlate with reduced snaps/production that week?

Also settles a documentation mismatch: the task that produced this feature
described a `date_modified` column as part of `load_injuries()`. It does
not exist -- not in nflreadpy's wrapper, not in the raw nflverse-data
release checked directly below. The honest substitute (the whole season
file's own last-regenerated timestamp) is what `get_injuries_source_
updated_at` surfaces instead -- file-level freshness, not per-row.

## Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import nflreadpy as nfl
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

from src.ingest import (
    get_injuries, get_injuries_source_updated_at, get_id_crosswalk,
    get_sleeper_players, get_sleeper_rosters, get_weekly_stats,
    get_snap_counts, DEFAULT_LEAGUE_ID,
)

SEASONS = list(range(2018, 2026))  # matches this project's default SEASONS

## 0. Schema check: no `date_modified`, no day-level practice data

Checked at both the nflreadpy layer and the raw nflverse-data release
(bypassing nflreadpy entirely) to make sure this isn't a wrapper-level
omission.

In [2]:
raw_2025 = nfl.load_injuries([2025]).to_pandas()
print("nflreadpy load_injuries() columns:")
print(list(raw_2025.columns))

direct = pd.read_parquet(
    "https://github.com/nflverse/nflverse-data/releases/download/injuries/injuries_2025.parquet"
)
print("\nRaw nflverse-data release columns (bypassing nflreadpy entirely):")
print(list(direct.columns))
print("\nSame column set:", set(raw_2025.columns) == set(direct.columns))
print("date_modified present anywhere:", "date_modified" in raw_2025.columns or "date_modified" in direct.columns)

nflreadpy load_injuries() columns:
['season', 'season_type', 'game_type', 'team', 'week', 'gsis_id', 'position', 'full_name', 'first_name', 'last_name', 'report_primary_injury', 'report_secondary_injury', 'report_status', 'practice_primary_injury', 'practice_secondary_injury', 'practice_status']

Raw nflverse-data release columns (bypassing nflreadpy entirely):
['season', 'season_type', 'game_type', 'team', 'week', 'gsis_id', 'position', 'full_name', 'first_name', 'last_name', 'report_primary_injury', 'report_secondary_injury', 'report_status', 'practice_primary_injury', 'practice_secondary_injury', 'practice_status']

Same column set: True
date_modified present anywhere: False


**Confirmed: no `date_modified` at any level of this source.** `practice_status`
is a single value per (player, week) -- there is no Wednesday/Thursday/Friday
breakdown to reshape into. The Practice Report tab presents both facts as
what they are: weekly, not daily; freshness-stamped at the file level via
`get_injuries_source_updated_at`, not per row.

In [3]:
season = 2026
updated_at = get_injuries_source_updated_at(season)
print(f"nflverse's injuries_{season} file was last regenerated: {updated_at}")

nflverse's injuries_2026 file was last regenerated: 2026-09-14T13:53:30Z


## 1. Does `practice_status`/`report_status` duplicate the Injury tab's `injury_status`?

Sleeper's `injury_status` (Q/D/O/IR/PUP/NA/SUSP) already drives the Injury
tab and Lineup Risks panel. Cross-referencing REAL rostered players in this
league's live 2026 week 1 report against Sleeper's own status for the same
players, via the DynastyProcess ID crosswalk.

In [4]:
crosswalk = get_id_crosswalk()
sleeper_players = get_sleeper_players()
rosters_raw = get_sleeper_rosters(DEFAULT_LEAGUE_ID)
rostered_ids = {pid for r in rosters_raw for pid in (r.get('players') or [])}

inj_2026 = get_injuries([2026])
inj_wk1 = inj_2026[inj_2026['week'] == 1]

cw = crosswalk[['sleeper_id', 'gsis_id']].dropna(subset=['gsis_id'])
merged = inj_wk1.merge(cw, on='gsis_id', how='inner')
merged = merged.merge(
    sleeper_players[['sleeper_id', 'injury_status']],
    on='sleeper_id', how='left',
)
merged['is_rostered'] = merged['sleeper_id'].isin(rostered_ids)
rostered_view = merged[merged['is_rostered']][
    ['full_name', 'position', 'report_status', 'injury_status', 'practice_status',
     'report_primary_injury', 'practice_primary_injury']
].reset_index(drop=True)
print(f"{len(rostered_view)} rostered players in this league have a real week-1 report:\n")
rostered_view

25 rostered players in this league have a real week-1 report:



,full_name,position,report_status,injury_status,practice_status,report_primary_injury,practice_primary_injury
0,Jeremiyah Love,RB,Questionable,NaN,Limited Participation in Practice,Ankle,Ankle
1,Zachariah Branch,WR,NaN,NaN,Full Participation in Practice,NaN,Hamstring
2,Zay Flowers,WR,NaN,Questionable,Full Participation in Practice,NaN,Hamstring
3,Jonathon Brooks,RB,NaN,NaN,Full Participation in Practice,NaN,Groin
4,Rome Odunze,WR,Questionable,NaN,Limited Participation in Practice,Calf,Calf
5,Tee Higgins,WR,NaN,NaN,Full Participation in Practice,NaN,Foot
6,Ja'Marr Chase,WR,NaN,NaN,Did Not Participate In Practice,NaN,Knee
7,Alec Pierce,WR,NaN,PUP,Limited Participation in Practice,NaN,Heel
8,Jakobi Meyers,WR,NaN,NaN,Limited Participation in Practice,NaN,Thumb
9,Patrick Mahomes,QB,NaN,Questionable,Full Participation in Practice,NaN,Knee


**They disagree, in both directions, often.** Real examples from the table
above (see the executed cell for the current live snapshot):

- nflverse's `report_status` already shows Questionable/Doubtful/Out for
  several rostered players whose Sleeper `injury_status` still reads
  healthy (`NaN`) -- Sleeper hasn't caught up to this week's official
  report yet.
- The reverse also happens: Sleeper flags a player Questionable while
  nflverse's `report_status` for that player is currently null (no
  official designation), even though `practice_status`/`practice_primary_
  injury` show a real, tracked issue.
- A player Sleeper marks `PUP` (a roster-level, off-season designation)
  can show up here with a genuine in-season `practice_status` (e.g.
  "Limited Participation") as they work back -- information the static
  PUP tag alone doesn't carry.

**Conclusion: these are complementary signals, not the same information
under two names.** Sleeper's `injury_status` is a roster-level flag
(covers IR/PUP/Suspended, which aren't weekly-report concepts at all, but
lags the live NFL report for in-season Q/D/O). nflverse's `report_status`/
`practice_status` is the authoritative CURRENT week's official report, but
carries no long-term roster designation. The Practice Report tab shows both
columns side by side rather than collapsing them -- a disagreement is
itself informative, not noise to hide.

## 2. `report_primary_injury` vs `practice_primary_injury` -- how often, and how, do they diverge?

In [5]:
inj_all = nfl.load_injuries(True).to_pandas()
both = inj_all.dropna(subset=['report_primary_injury', 'practice_primary_injury'])
diff = both[both['report_primary_injury'].str.lower() != both['practice_primary_injury'].str.lower()]
print(f"Rows with BOTH populated: {len(both):,}")
print(f"Rows where they DIFFER: {len(diff):,} ({len(diff) / len(both) * 100:.2f}%)")
diff[['season', 'week', 'team', 'full_name', 'report_primary_injury', 'practice_primary_injury', 'report_status']].tail(15)

Rows with BOTH populated: 60,310
Rows where they DIFFER: 123 (0.20%)


,season,week,team,full_name,report_primary_injury,practice_primary_injury,report_status
84335,2024.0,18.0,PHI,Saquon Barkley,Not injury related - coach's decision,Not injury related - resting player,Doubtful
84336,2024.0,18.0,PHI,Zack Baun,Not injury related - coach's decision,Not injury related - resting player,Doubtful
84337,2024.0,18.0,PHI,Jalen Carter,Not injury related - coach's decision,Not injury related - resting player,Doubtful
84338,2024.0,18.0,PHI,Landon Dickerson,Not injury related - coach's decision,Not injury related - resting player,Doubtful
84339,2024.0,18.0,PHI,Lane Johnson,Not injury related - coach's decision,Not injury related - resting player,Doubtful
84340,2024.0,18.0,PHI,Jordan Mailata,Not injury related - coach's decision,Not injury related - resting player,Doubtful
84341,2024.0,18.0,PHI,Darius Slay,Not injury related - coach's decision,Not injury related - resting player,Doubtful
86066,2025.0,6.0,CIN,Ja'Marr Chase,Illness,Not injury related - resting player,Questionable
86213,2025.0,6.0,NE,Milton Williams,Illness,Ankle,Questionable
88618,2025.0,13.0,WAS,Chris Moore,Illness,Shoulder,Questionable


**Rare (about 1 in 500 report-rows) but a real, useful signal when it happens.**
The dominant pattern: an **illness** is what's actually driving that week's
official game designation, layered on top of a separately-tracked physical
injury that's what shows up in the practice report (e.g. Ja'Marr Chase,
2025 Week 6: practice report says "Not injury related - resting player",
but the official designation is driven by "Illness"). A smaller pattern
(2024 Week 18 Eagles) is pure roster management -- "coach's decision" vs.
"resting player" for seven players resting ahead of the playoffs, not an
injury discrepancy at all.

**Design decision this drove:** the Practice Report tab shows
`practice_primary_injury` as the main Injury value (it's populated far more
often and describes what's actually being managed day to day), and adds a
small "Official report: X" annotation ONLY when `report_primary_injury` is
present and names a genuinely different issue -- not a separate column
that would be blank 99.8% of the time.

## 3. Is practice participation actually predictive?

Two real, separate questions:
- Does the practice designation predict whether the player suits up at all?
- CONDITIONAL on playing, does it predict reduced production or snap share?

Scope: QB/RB/WR/TE, 2018-2025 (this project's default `SEASONS`), REG season
only (`game_type == 'REG'` -- `season_type` is only populated for 2025-2026
in this source, checked directly, so `game_type` is the reliable filter).

In [6]:
inj = inj_all[(inj_all['season'].isin(SEASONS)) & (inj_all['game_type'] == 'REG')]
inj = inj[inj['position'].isin(['QB', 'RB', 'WR', 'TE'])].copy()
inj['season'] = inj['season'].astype(int)
inj['week'] = inj['week'].astype(int)
print(f"{len(inj):,} skill-position injury-report rows, {SEASONS[0]}-{SEASONS[-1]} REG season")
inj['practice_status'].value_counts(dropna=False)

13,375 skill-position injury-report rows, 2018-2025 REG season


practice_status
Full Participation in Practice       6206
Did Not Participate In Practice      3681
Limited Participation in Practice    3406
\n                                     67
NaN                                    15
Name: count, dtype: int64

In [7]:
weekly = get_weekly_stats(SEASONS)
weekly = weekly[weekly['season_type'] == 'REG'][
    ['player_id', 'season', 'week', 'fantasy_points_ppr']
].rename(columns={'player_id': 'gsis_id'}).copy()
weekly['season'] = weekly['season'].astype(int)
weekly['week'] = weekly['week'].astype(int)

snaps = get_snap_counts(SEASONS)
snaps = snaps[snaps['game_type'] == 'REG'][['pfr_player_id', 'season', 'week', 'offense_pct']].copy()
cw_pfr = get_id_crosswalk()[['gsis_id', 'pfr_id']].dropna()
snaps = snaps.merge(cw_pfr, left_on='pfr_player_id', right_on='pfr_id', how='inner')
snaps['season'] = snaps['season'].astype(int)
snaps['week'] = snaps['week'].astype(int)
snaps = snaps.groupby(['gsis_id', 'season', 'week'], as_index=False)['offense_pct'].max()

full = weekly.merge(
    inj[['gsis_id', 'season', 'week', 'practice_status', 'report_status']],
    on=['gsis_id', 'season', 'week'], how='outer', indicator=True,
).merge(snaps, on=['gsis_id', 'season', 'week'], how='left')

on_report = full[full['practice_status'].notna()].copy()
on_report['played'] = on_report['_merge'].isin(['both', 'left_only'])
print(f"{len(on_report):,} skill-position player-weeks on the injury report")

13,360 skill-position player-weeks on the injury report


### 3a. Play rate by practice_status

In [8]:
played_rate = on_report.groupby('practice_status')['played'].agg(['mean', 'count']).sort_values('mean')
played_rate.columns = ['play_rate', 'n_player_weeks']
played_rate

,play_rate,n_player_weeks
practice_status,,
Did Not Participate In Practice,0.158381,3681
\n,0.447761,67
Limited Participation in Practice,0.623018,3406
Full Participation in Practice,0.835965,6206


### 3b. Play rate by the OFFICIAL report_status designation

In [9]:
on_report['report_status_filled'] = on_report['report_status'].fillna('(no designation)')
report_play_rate = on_report.groupby('report_status_filled')['played'].agg(['mean', 'count']).sort_values('mean')
report_play_rate.columns = ['play_rate', 'n_player_weeks']
report_play_rate

,play_rate,n_player_weeks
report_status_filled,,
Out,0.000389,2571
Doubtful,0.007371,407
Questionable,0.583633,3336
(no designation),0.847530,7044
Note,1.000000,2


### 3c. Conditional on playing: production and snap-share vs. the player's own healthy-week baseline

Comparing each on-report played week to that SAME player's own median on
weeks with NO injury-report entry at all that season -- controls for skill
differences across players (a raw pooled average would just measure "which
players get hurt," not "what does being hurt cost").

In [10]:
healthy_weeks = full[full['practice_status'].isna() & (full['_merge'] == 'left_only')]
baseline_ppr = healthy_weeks.groupby('gsis_id')['fantasy_points_ppr'].median().rename('baseline_ppr')
baseline_snap = healthy_weeks.groupby('gsis_id')['offense_pct'].median().rename('baseline_snap_pct')

played = on_report[on_report['played']].merge(baseline_ppr, on='gsis_id', how='left').merge(baseline_snap, on='gsis_id', how='left')
played = played[played['baseline_ppr'] > 1.0]  # exclude players with no real fantasy-relevant baseline
played['ppr_ratio'] = played['fantasy_points_ppr'] / played['baseline_ppr']
played['snap_ratio'] = played['offense_pct'] / played['baseline_snap_pct']

print("Fantasy-points ratio vs. own healthy-week baseline (played weeks only):")
display(played.groupby('practice_status')['ppr_ratio'].agg(['median', 'mean', 'count']))

print("\nSnap-share ratio vs. own healthy-week baseline (played weeks only, has snap data):")
snap_valid = played.dropna(subset=['snap_ratio', 'baseline_snap_pct'])
snap_valid = snap_valid[snap_valid['baseline_snap_pct'] > 0.1]
display(snap_valid.groupby('practice_status')['snap_ratio'].agg(['median', 'mean', 'count']))

Fantasy-points ratio vs. own healthy-week baseline (played weeks only):


,median,mean,count
practice_status,,,
\n,0.927341,1.445246,28
Did Not Participate In Practice,0.826923,1.050234,555
Full Participation in Practice,1.000000,1.280558,4812
Limited Participation in Practice,0.945455,1.239058,2006



Snap-share ratio vs. own healthy-week baseline (played weeks only, has snap data):


,median,mean,count
practice_status,,,
\n,1.051671,1.068040,28
Did Not Participate In Practice,0.953935,0.953555,554
Full Participation in Practice,1.000000,1.030502,4788
Limited Participation in Practice,0.968404,0.977196,1992


## Findings, stated plainly

**Yes -- practice participation is measurably predictive, not just informational.**

1. **Play rate tracks practice status monotonically**: Did Not Participate
   -> ~16% play rate that week; Limited -> ~62%; Full -> ~84%. A DNP
   designation is a real, strong (if imperfect) signal the player won't
   suit up.
2. **The OFFICIAL `report_status` is a much sharper "will they play"
   signal than practice status alone**: Out/Doubtful -> essentially never
   play (<1%); Questionable -> a real coin flip (~58%); no designation ->
   normal play rate (~85%). This matches -- and meaningfully revises -- the
   Lineup Risks panel's existing hand-picked heuristics
   (`playProbability()` in index.html), which currently assume ~75% for
   Questionable and ~25% for Doubtful. The measured numbers here are
   materially different (58% and <1%). Reported here, not wired in --
   updating that panel's heuristic is a separate, deliberate decision this
   notebook doesn't make.
3. **Conditional on playing anyway, there's a real (if modest) production
   discount**: Limited-but-played weeks land at ~94-95% of the player's own
   healthy-week median (confirmed independently via fantasy points AND
   snap share -- two different measures agreeing is what makes this a real
   finding, not noise from one metric). A DNP-but-played week's discount is
   roughly triple that.

**This finding is reported, not wired into the projection model** -- per
this task's own scope and CLAUDE.md's model-scope boundary (tabular
regression on real usage/efficiency features, not a practice-report
input). It's real enough to be worth a future, deliberate feature-family
proposal (Family 5B/Team Tendencies/Context Columns' own precedent for how
this project vets a new feature family before wiring it in), not
something this notebook decides on its own.